In [233]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from rdkit import Chem
from modules.data_loader import *

In [234]:
visco = pd.read_csv(DATA_DIR / 'Viscoelastic_WLM.csv')
visco.head()

,Source,ID_surfactant1,Type_surfactant1,SMILES_surfactant1,C_surfactant1 (mM),ID_surfactant2,Type_surfactant2,SMILES_surfactant2,C_surfactant2 (mM),ID_additive1,...,ID_additive2,SMILES_additive2,C_additive2 (mM),T [°C],Viscoelasticity,G₀ [Pa],"G""min [Pa]",ω* cr [rad/s],ͳR [s],η₀[Pa∙s]
0,A1,SLES,anionic,CCCCCCCCCCCCOCCOS(=O)(=O)[O-].[Na+],395.0,LED1A,zwitterionic,CCCCCCCCCCCC(=O)NCCCN+(C)C([O-])=O,55.0,NaCl,...,NaN,NaN,NaN,25,0.0,NaN,NaN,NaN,NaN,0.02
1,A1,SLES,anionic,CCCCCCCCCCCCOCCOS(=O)(=O)[O-].[Na+],395.0,LED1A,zwitterionic,CCCCCCCCCCCC(=O)NCCCN+(C)C([O-])=O,55.0,NaCl,...,NaN,NaN,NaN,25,1.0,4.0,2.70,150.0,0.01,0.03
2,A1,SLES,anionic,CCCCCCCCCCCCOCCOS(=O)(=O)[O-].[Na+],395.0,LED1A,zwitterionic,CCCCCCCCCCCC(=O)NCCCN+(C)C([O-])=O,55.0,NaCl,...,NaN,NaN,NaN,25,1.0,4.5,1.20,10.0,0.10,0.45
3,A1,SLES,anionic,CCCCCCCCCCCCOCCOS(=O)(=O)[O-].[Na+],395.0,LED1A,zwitterionic,CCCCCCCCCCCC(=O)NCCCN+(C)C([O-])=O,55.0,NaCl,...,NaN,NaN,NaN,25,1.0,5.0,0.50,0.8,1.25,6.25
4,A1,SLES,anionic,CCCCCCCCCCCCOCCOS(=O)(=O)[O-].[Na+],395.0,LED1A,zwitterionic,CCCCCCCCCCCC(=O)NCCCN+(C)C([O-])=O,55.0,NaCl,...,NaN,NaN,NaN,25,1.0,4.2,0.35,1.5,0.67,2.80


## Renaming columns

In [235]:
d = {
    'Source': 'source',
    'ID_surfactant1' : 'id_surfactant1',
    'Type_surfactant1' : 'type_surfactant1',
    'SMILES_surfactant1' : 'smiles_surfactant1',
    'C_surfactant1 (mM)' : 'c_surfactant1',
    'ID_surfactant2' : 'id_surfactant2',
    'Type_surfactant2' : 'type_surfactant2',
    'SMILES_surfactant2' : 'smiles_surfactant2',
    'C_surfactant2 (mM)' : 'c_surfactant2',
    'ID_additive1' : 'id_additive1',
    'SMILES_additive1' : 'smiles_additive1',
    'C_additive1 (mM)' : 'c_additive1',
    'ID_additive2' : 'id_additive2',
    'SMILES_additive2' : 'smiles_additive2',
    'C_additive2 (mM)' : 'c_additive2',
    'T [°C]' : 'temperature',
    'Viscoelasticity' : 'viscoelasticity',
    'G₀  [Pa]' : 'g_0',
    'G"min [Pa]' : 'g_min',
    'ω* cr [rad/s]' : 'omega_cr',
    'ͳR [s]' : 'tau_r',
    'η₀[Pa∙s]' : 'eta_0'
}

visco = visco.rename(columns = d)

## Replacing NaNs

In [236]:
visco = visco.dropna(subset=['viscoelasticity'])

In [237]:
visco[['c_surfactant1', 'c_surfactant2', 'c_additive1', 'c_additive2']] = visco[['c_surfactant1', 'c_surfactant2', 'c_additive1', 'c_additive2']].fillna(0)

In [238]:
visco.info()

<class 'pandas.DataFrame'>
Index: 8276 entries, 0 to 8394
Data columns (total 22 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   source              8276 non-null   str    
 1   id_surfactant1      8276 non-null   str    
 2   type_surfactant1    8276 non-null   str    
 3   smiles_surfactant1  8276 non-null   str    
 4   c_surfactant1       8276 non-null   float64
 5   id_surfactant2      2774 non-null   str    
 6   type_surfactant2    2774 non-null   str    
 7   smiles_surfactant2  2679 non-null   str    
 8   c_surfactant2       8276 non-null   float64
 9   id_additive1        5610 non-null   str    
 10  smiles_additive1    5610 non-null   str    
 11  c_additive1         8276 non-null   float64
 12  id_additive2        1295 non-null   str    
 13  smiles_additive2    1139 non-null   str    
 14  c_additive2         8276 non-null   float64
 15  temperature         8276 non-null   int64  
 16  viscoelasticity     82

## Removing erroneous samples

Based on calculated/provided eta0 mismatch

In [239]:
visco = visco.query("source != 'A81'")
visco = visco.query("source != 'A118'")
visco = visco.query("source != 'A85'")

## Correcting SMILES typos

In [240]:
def validate_smiles(smiles):
    if pd.isna(smiles):
        return True
    mol = Chem.MolFromSmiles(str(smiles))
    if mol is None:
        return False
    return True

def correct_smiles(smiles):
    if smiles == 'CCCCCCCCCCCCCCCCOCC(O)CN+(C)C.[Br-]':
        return 'CCCCCCCCCCCCCCCCOCC(O)C[N+](C)C.[Br-]'
    elif smiles == 'CCCCCCCCCCCCCCCCN+(C)C.[Cl-]':
        return 'CCCCCCCCCCCCCCCC[N+](C)C.[Cl-]'
    elif smiles == 'CCCCCCCCCCCCCCCCN+(C)C.[Br-]':
        return 'CCCCCCCCCCCCCCCC[N+](C)C.[Br-]'
    elif smiles == 'CCCCCCCCCCCC(=O)NCCCN+(C)C([O-])=O':
        return 'CCCCCCCCCCCC(=O)NCCC[N+](C)C([O-])=O'
    elif smiles == 'CCCCCCCCCCCC(=O)NCCCN+(C)(C)CC([O-])=O':
        return 'CCCCCCCCCCCC(=O)NCCC[N+](C)(C)CC([O-])=O'
    elif smiles == '[Al+3].[O-]N+[O-].[O-]N+[O-].[O-]N+[O-]':
        return '[Al+3].[O-][N+][O-].[O-][N+][O-].[O-][N+][O-]'
    elif smiles == '[Na+].[O-]N+[O-]':
        return '[Na+].[O-][N+][O-]'
    elif smiles == 'CCCCCC1C(CCCCCC(=O)NCCCN(C)(C)CC(=O)[O-])C(C=CC1CCCCCC)CCCCCC(=O)NCCCN(C)(C)CC(=O)[O-]':
        return 'CCCCCCCCCCCC[N+](C)(C)CC(=O)[O-]'
    return smiles

In [241]:
visco['smiles_surfactant1'] = visco['smiles_surfactant1'].apply(correct_smiles)
visco['smiles_surfactant2'] = visco['smiles_surfactant2'].apply(correct_smiles)
visco['smiles_additive1'] = visco['smiles_additive1'].apply(correct_smiles)
visco['smiles_additive2'] = visco['smiles_additive2'].apply(correct_smiles)

In [242]:
smiles = list(visco['smiles_surfactant1'].unique())
smiles.extend(visco['smiles_surfactant2'].unique())
smiles.extend(visco['smiles_additive1'].unique())
smiles.extend(visco['smiles_additive2'].unique())
len(smiles)
for smile in smiles:
    if not validate_smiles(smile):
        print(smile)

In [243]:
visco.info()

<class 'pandas.DataFrame'>
Index: 8214 entries, 0 to 8394
Data columns (total 22 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   source              8214 non-null   str    
 1   id_surfactant1      8214 non-null   str    
 2   type_surfactant1    8214 non-null   str    
 3   smiles_surfactant1  8214 non-null   str    
 4   c_surfactant1       8214 non-null   float64
 5   id_surfactant2      2731 non-null   str    
 6   type_surfactant2    2731 non-null   str    
 7   smiles_surfactant2  2636 non-null   str    
 8   c_surfactant2       8214 non-null   float64
 9   id_additive1        5548 non-null   str    
 10  smiles_additive1    5548 non-null   str    
 11  c_additive1         8214 non-null   float64
 12  id_additive2        1279 non-null   str    
 13  smiles_additive2    1123 non-null   str    
 14  c_additive2         8214 non-null   float64
 15  temperature         8214 non-null   int64  
 16  viscoelasticity     82

## SMILES to canonical

In [244]:
def smiles_to_canonical(smiles):
    if pd.isna(smiles):
        return smiles
    return Chem.MolToSmiles(Chem.MolFromSmiles(smiles))

In [245]:
columns = ['smiles_surfactant1', 'smiles_surfactant2','smiles_additive1', 'smiles_additive2']
visco[columns] = visco[columns].apply(lambda col: col.map(smiles_to_canonical))

In [246]:
visco.duplicated().sum()

np.int64(29)

In [247]:
visco = visco.drop_duplicates(subset=visco.columns.difference(['source']))

## Removing erroneous samples

Based on calculated/provided eta0 mismatch (absolute and percentage mismatch)

In [248]:
outliers = pd.read_csv(DATA_DIR / 'outliers.csv')
merged = pd.merge(visco, outliers, how='left', indicator=True)
clean_data = merged.query("_merge == 'left_only'").drop(columns='_merge')

csv_count = len(outliers)
removed_count = len(visco) - len(clean_data)
assert csv_count == removed_count

## Saving features

In [249]:
save_features(clean_data, 'viscoelastic_cleaned')